# TensorFlow Recommenders on MovieLens-1M

Two standard [TensorFlow Recommenders (TFRS)](https://www.tensorflow.org/recommenders) retrieval models
(`genrec.models.tfrs_models`), evaluated exactly like every other model:

| model | query tower ("who is asking") | order of ratings |
|---|---|---|
| **TFRS two-tower** | a learned vector per user (user-ID embedding) | ignored - like collaborative filtering, but learned |
| **TFRS sequential (GRU, last 20)** | a GRU reading the user's last 20 movies | used - comparable to SASRec-20 and the Qwen prompt |

Both are trained with TFRS's `Retrieval` task (in-batch softmax). Same split, the same 1,000 test users, full
ranking, already-rated movies skipped. The number of epochs is picked on the validation item.

**Runs in the separate TensorFlow environment** (kernel *Python (.venv-tf)*, see the README). CPU only, ~15 min.
TensorFlow training on CPU is not bit-for-bit repeatable: expect differences of ~0.01 HR@10 between runs.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time

import pandas as pd
import tensorflow as tf
import tf_keras as keras

from genrec.data import Dataset, history, sample_test_users, target
from genrec.metrics import evaluate_scores, top_k
from genrec.models import tfrs_models as T
from genrec.paths import ML1M_DIR, RESULTS_DIR

MAX_EPOCHS = 30
PATIENCE = 3       # stop when validation NDCG@10 hasn't improved for this many epochs
CONTEXT = 20
SEED = 42
print(f"tensorflow {tf.__version__}, tf_keras {keras.__version__}")

tensorflow 2.21.0, tf_keras 2.21.0


## 1. Data

In [2]:
ds = Dataset.load(ML1M_DIR)
seqs = ds.encoded()
users = ds.users
test_users = sample_test_users(users)
row_of = {u: r for r, u in enumerate(users)}
title = dict(zip(range(1, ds.index.n_cols), ds.movies.set_index("MovieID").loc[ds.index.movie_ids, "Title"]))


def split_lists(us, split):
    return [history(seqs[u], split) for u in us], [target(seqs[u], split) for u in us]


val_hist, val_tgt = split_lists(users, "val")


def fit_best(model, train_ds, score_fn, name):
    """Train epoch by epoch, keep the weights with the best validation NDCG@10 (all users)."""
    best, best_epoch, best_weights, t0 = -1.0, 0, None, time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        hist = model.fit(train_ds, epochs=1, verbose=0)
        val = evaluate_scores(score_fn(users, val_hist), val_hist, val_tgt)
        if val["NDCG@10"] > best:
            best, best_epoch, best_weights = val["NDCG@10"], epoch, model.get_weights()
        print(f"{name} epoch {epoch:2d}  loss {hist.history['loss'][0]:9.1f}  val HR@10 {val['HR@10']:.3f}  "
              f"NDCG@10 {val['NDCG@10']:.3f}  (best {best:.3f} @ {best_epoch})  {time.time() - t0:.0f}s")
        if epoch - best_epoch >= PATIENCE:
            break
    model.set_weights(best_weights)
    return best_epoch

## 2. Two-tower (user ID -> movie)

One embedding per user, so it only knows what the user rated during training: validation trains on `items[:-2]`,
the test model is retrained on `items[:-1]` for the number of epochs that was best on validation.

In [3]:
def two_tower():
    tf.random.set_seed(SEED)
    model = T.TwoTower(len(users), ds.index.n_cols)
    model.compile(optimizer=keras.optimizers.Adagrad(learning_rate=0.1))
    return model


twotower = two_tower()
best_epochs = fit_best(twotower, T.pairs_dataset(range(len(users)), val_hist),
                       lambda us, _: twotower.scores([row_of[u] for u in us]), "two-tower")
twotower = two_tower()
twotower.fit(T.pairs_dataset(range(len(users)), split_lists(users, "test")[0]), epochs=best_epochs, verbose=0)
print(f"two-tower retrained on items[:-1] for {best_epochs} epochs")

two-tower epoch  1  loss    6368.2  val HR@10 0.049  NDCG@10 0.023  (best 0.023 @ 1)  8s


two-tower epoch  2  loss    6245.5  val HR@10 0.053  NDCG@10 0.025  (best 0.025 @ 2)  16s


two-tower epoch  3  loss    6179.2  val HR@10 0.056  NDCG@10 0.026  (best 0.026 @ 3)  23s


two-tower epoch  4  loss    6134.4  val HR@10 0.056  NDCG@10 0.026  (best 0.026 @ 3)  30s


two-tower epoch  5  loss    6101.4  val HR@10 0.053  NDCG@10 0.025  (best 0.026 @ 3)  38s


two-tower epoch  6  loss    6075.7  val HR@10 0.052  NDCG@10 0.024  (best 0.026 @ 3)  45s


two-tower retrained on items[:-1] for 3 epochs


## 3. Sequential: GRU over the last 20 movies -> next movie

Reads any history at prediction time, so no retraining for the test (like SASRec). Trained on every position of
`items[:-2]`.

In [4]:
tf.random.set_seed(SEED)
sequential = T.SequentialRetrieval(ds.index.n_cols, context=CONTEXT)
sequential.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
train_ds, n_examples = T.sequence_dataset(val_hist, CONTEXT)
print(f"{n_examples} training examples")
fit_best(sequential, train_ds, lambda _, hist: sequential.scores(hist), "sequential")

982089 training examples


sequential epoch  1  loss     214.6  val HR@10 0.126  NDCG@10 0.065  (best 0.065 @ 1)  41s


sequential epoch  2  loss     182.5  val HR@10 0.165  NDCG@10 0.088  (best 0.088 @ 2)  80s


sequential epoch  3  loss     167.6  val HR@10 0.180  NDCG@10 0.096  (best 0.096 @ 3)  119s


sequential epoch  4  loss     160.9  val HR@10 0.187  NDCG@10 0.100  (best 0.100 @ 4)  156s


...

sequential epoch 10  loss     143.2  val HR@10 0.200  NDCG@10 0.109  (best 0.109 @ 10)  394s


sequential epoch 11  loss     143.0  val HR@10 0.200  NDCG@10 0.108  (best 0.109 @ 10)  433s


sequential epoch 12  loss     142.8  val HR@10 0.201  NDCG@10 0.108  (best 0.109 @ 10)  473s


sequential epoch 13  loss     142.0  val HR@10 0.201  NDCG@10 0.108  (best 0.109 @ 10)  513s


10

## 4. Test-set comparison

In [5]:
def test_scores(us):
    hist, _ = split_lists(us, "test")
    return {"TFRS two-tower": twotower.scores([row_of[u] for u in us]),
            f"TFRS sequential (GRU, last {CONTEXT})": sequential.scores(hist)}


test_hist, test_tgt = split_lists(test_users, "test")
scores = test_scores(test_users)
table = pd.DataFrame({name: evaluate_scores(s, test_hist, test_tgt) for name, s in scores.items()}).T
RESULTS_DIR.mkdir(exist_ok=True)
table.to_csv(RESULTS_DIR / "tfrs_test.csv")   # picked up by baselines.ipynb
print(f"test: last rating of {len(test_users)} users")
table.style.format("{:.3f}")

test: last rating of 1000 users


,hit@1,HR@10,NDCG@10
TFRS two-tower,0.008,0.055,0.027
"TFRS sequential (GRU, last 20)",0.029,0.173,0.092


### All 6,040 users

In [6]:
all_hist, all_tgt = split_lists(users, "test")
pd.DataFrame({name: evaluate_scores(s, all_hist, all_tgt) for name, s in test_scores(users).items()}).T \
    .style.format("{:.3f}")

,hit@1,HR@10,NDCG@10
TFRS two-tower,0.006,0.052,0.025
"TFRS sequential (GRU, last 20)",0.038,0.183,0.100


## 5. Example recommendations

In [7]:
for r, u in enumerate(test_users[:3]):
    hist, tgt = test_hist[r], test_tgt[r]
    print(f"user {u}  last rated: {'; '.join(title[m] for m in hist[-5:])}")
    print(f"   real next: {title[tgt]}")
    for name, s in scores.items():
        top = top_k(s[r:r + 1], [hist])[0].tolist()
        verdict = f"HIT at rank {top.index(tgt) + 1}" if tgt in top else "miss"
        print(f"   {name[:15]:15s} ({verdict}): {'; '.join(title[m] for m in top[:5])} ...")
    print()

user 3156  last rated: Gone in 60 Seconds (2000); Almost Famous (2000); Best in Show (2000); Tao of Steve, The (2000); Meet the Parents (2000)
   real next: Space Cowboys (2000)
   TFRS two-tower  (miss): Shaft (2000); In Too Deep (1999); Romeo Must Die (2000); World Is Not Enough, The (1999); Art of War, The (2000) ...
   TFRS sequential (miss): Nurse Betty (2000); Original Kings of Comedy, The (2000); Barenaked in America (1999); Contender, The (2000); Woman on Top (2000) ...

user 3446  last rated: Frighteners, The (1996); Fright Night (1985); Phantasm IV: Oblivion (1998); Nightmare on Elm Street, A (1984); Blob, The (1958)
   real next: Near Dark (1987)
   TFRS two-tower  (HIT at rank 3): Blood Feast (1963); Two Thousand Maniacs! (1964); Near Dark (1987); Wolf Man, The (1941); Masque of the Red Death, The (1964) ...
   TFRS sequential (miss): Inferno (1980); Hunger, The (1983); Gremlins (1984); Howling, The (1980); Lifeforce (1985) ...

user 332  last rated: Annie Hall (1977); Home